In [ ]:
import pandas as pd
import numpy as np
import sys
import os

# Add the src directory to Python path
# From fig_4_d/ go up 5 levels to reach src/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', '..', '..', '..'))
src_path = os.path.join(project_root, 'src')
sys.path.insert(0, src_path)

from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from mimiciii_db import DB
from mimiciii_db.config import db_url


In [ ]:
db = DB.from_url(db_url())
print("Database connected successfully!")


In [ ]:
lca_path = os.getenv("LCA_PATH")
lca = pd.read_csv(lca_path)

In [ ]:
filtered_patients_with_morbidity_counts_path = os.getenv("FILTERED_PATIENTS_WITH_MORBIDITY_COUNTS_PATH")
filtered_patients_with_morbidity_counts = db.table_df(filtered_patients_with_morbidity_counts_path, schema="mimiciii")

In [ ]:
# --- Load SOFA and OASIS data
sofa_path = os.getenv("SOFA_PATH")
oasis_path = os.getenv("OASIS_PATH")
sofa_df  = pd.read_csv(sofa_path)
oasis_df = pd.read_csv(oasis_path)

patients = filtered_patients_with_morbidity_counts.copy()

# --- Ensure key dtypes match
keys = ['subject_id','hadm_id']
for df in (patients, sofa_df, oasis_df):
    for k in keys:
        df[k] = pd.to_numeric(df[k], errors='coerce')

# keep first; or replace with .groupby(keys).agg({'sofa_total':'max', ...})
sofa_df  = sofa_df.drop_duplicates(subset=keys)
oasis_df = oasis_df.drop_duplicates(subset=keys)

# --- Left join SOFA then OASIS
merged = (
    patients
      .merge(sofa_df,  on=keys, how='left', suffixes=('', '_sofa'))
      .merge(oasis_df, on=keys, how='left', suffixes=('', '_oasis'))
)

print("patients rows:", len(patients))
print("merged rows  :", len(merged))


In [ ]:
df_merged = merged.merge(lca[['hadm_id','latent_class']],
                            on='hadm_id', how='left')


In [ ]:
# Load angus/sepsis data
angus = db.query_df("SELECT * FROM angus")


In [ ]:
admin = angus.copy()  
keys = ['subject_id','hadm_id']

for df_ in (df_merged, admin):
    for k in keys:
        df_[k] = pd.to_numeric(df_[k], errors='coerce')

admin = admin.drop_duplicates(subset=keys)           # 1 row per (subject, hadm)
dfA = df_merged.merge(admin, on=keys, how='left')


In [ ]:
# Create sepsis column from angus if it doesn't exist
if 'sepsis' not in dfA.columns and 'angus' in dfA.columns:
    dfA['sepsis'] = dfA['angus']

# Create mortality column from hospital_expire_flag if it doesn't exist
if 'mortality' not in dfA.columns and 'hospital_expire_flag' in dfA.columns:
    dfA['mortality'] = dfA['hospital_expire_flag']

# make sure 0/1 and subgroup are numeric
for c in ['organ_dysfunction','sepsis','mortality']:
    dfA[c] = pd.to_numeric(dfA[c], errors='coerce')

dfA['latent_class'] = pd.to_numeric(dfA['latent_class'], errors='coerce').astype('Int64')
dfA = dfA[dfA['latent_class'].between(1,6)].copy()


In [ ]:
import matplotlib.pyplot as plt

df = dfA.drop_duplicates(['subject_id','hadm_id']).copy()
df['latent_class'] = pd.to_numeric(df['latent_class'], errors='coerce').astype('Int64')
df = df[df['latent_class'].between(1,6)].copy()

# ensure 0/1 ints
for c in ['organ_dysfunction','sepsis','mortality']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)

# Wilson CI for proportions
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n
    den = 1 + z**2/n
    cen = (p + z**2/(2*n))/den
    half = z*np.sqrt((p*(1-p) + z**2/(4*n))/n)/den
    return cen-half, cen+half

def conditional_mort(flag_col):
    # compute mortality ONLY among rows with flag==1, by subgroup
    groups = []
    for sg, sub in df.groupby('latent_class'):
        sub = sub[sub[flag_col] == 1]
        n = len(sub)
        k = sub['mortality'].sum()
        lo, hi = wilson_ci(k, n)
        groups.append({'subgroup': int(sg),
                       'n': n,
                       'p': 100*(k/n) if n>0 else np.nan,
                       'err_low': 100*((k/n)-lo) if n>0 else np.nan,
                       'err_high': 100*(hi-(k/n)) if n>0 else np.nan})
    out = pd.DataFrame(groups).sort_values('subgroup')
    return out

mort_od  = conditional_mort('organ_dysfunction')   # mortality among organ-dysfunction
mort_sep = conditional_mort('sepsis')        # mortality among sepsis

# colors per subgroup (same color for the pair) - matching paper palette
colors = {1:'#FFFFFF', 2:'#E41A1C', 3:'#4DAF4A', 4:'#377EB8', 5:'#4DD2D2', 6:'#E377C2'}

# ---- Panel D: two bars per subgroup (OD vs Sepsis), same color ----
x = mort_od['subgroup'].to_numpy()
w = 0.32

fig, ax = plt.subplots(figsize=(6,4))
for i, sg in enumerate(x):
    col = colors[sg]
    # left bar: organ dysfunction mortality
    ax.bar(sg - w/2, mort_od.loc[mort_od['subgroup']==sg, 'p'],
           width=w,
           yerr=np.vstack([mort_od.loc[mort_od['subgroup']==sg, 'err_low'],
                           mort_od.loc[mort_od['subgroup']==sg, 'err_high']]).astype(float),
           capsize=3, edgecolor='black', linewidth=1.0, color=col)
    # right bar: sepsis mortality
    ax.bar(sg + w/2, mort_sep.loc[mort_sep['subgroup']==sg, 'p'],
           width=w,
           yerr=np.vstack([mort_sep.loc[mort_sep['subgroup']==sg, 'err_low'],
                           mort_sep.loc[mort_sep['subgroup']==sg, 'err_high']]).astype(float),
           capsize=3, edgecolor='black', linewidth=1.0, color=col)

ax.set_xlabel('Subgroup')
ax.set_ylabel('Percent mortality')
ax.set_xticks(x)
ax.set_ylim(0, 40)       # match paper scale
ax.set_title('D')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, linestyle=':', linewidth=0.7, alpha=0.6)
plt.tight_layout()

# Save figure
import os
os.makedirs('/Users/gloriaye/Desktop/dsc180ab/25fa-dsc180a-team1/assets/fig_4', exist_ok=True)
plt.savefig("../assets/fig_4/fig_4d.png")
plt.show()
